# 00 — Prepare Data & Lock Master Split

Chạy notebook này **một lần duy nhất**.

Notebook:
- mount Google Drive;
- upload `gold_standard_final_48670.csv`;
- kiểm tra dataset;
- tính SHA-256;
- tạo **một master split 80/10/10 duy nhất**;
- stratify theo `NON_TOXIC / TOXIC_EXPLICIT / TOXIC_IMPLICIT`;
- lưu `train.csv`, `dev.csv`, `test.csv`, `split_manifest.csv`.

Các notebook model **không được tự split lại dữ liệu**.

In [ ]:
!pip -q install "pandas==2.2.2" "scikit-learn==1.6.1"

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")

In [ ]:
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

ROOT = Path("/content/drive/MyDrive/DS108_Toxic_4Models")
DATA_DIR = ROOT / "data"
SPLIT_DIR = ROOT / "data_splits"
RESULT_DIR = ROOT / "results"

for p in [DATA_DIR, SPLIT_DIR, RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

In [ ]:
DATASET_PATH = DATA_DIR / "gold_standard_final_48670.csv"

if not DATASET_PATH.exists():
    uploaded = files.upload()
    if "gold_standard_final_48670.csv" in uploaded:
        name = "gold_standard_final_48670.csv"
    else:
        csvs = [x for x in uploaded if x.lower().endswith(".csv")]
        if len(csvs) != 1:
            raise RuntimeError("Hãy upload đúng file gold_standard_final_48670.csv")
        name = csvs[0]
    DATASET_PATH.write_bytes(uploaded[name])

DATASET_SHA256 = sha256_file(DATASET_PATH)
print("Dataset:", DATASET_PATH)
print("SHA256:", DATASET_SHA256)

In [ ]:
df = pd.read_csv(DATASET_PATH)

required = ["row_id", "comment", "toxicity", "expression_type", "evidence"]
assert df.columns.tolist() == required, df.columns.tolist()
assert len(df) == 48670
assert df["row_id"].notna().all() and df["row_id"].is_unique
assert df["comment"].notna().all() and df["comment"].is_unique

assert set(df["toxicity"].unique()) == {"NON-TOXIC", "TOXIC"}
is_toxic = df["toxicity"].eq("TOXIC")
assert set(df.loc[is_toxic, "expression_type"].unique()) == {"EXPLICIT", "IMPLICIT"}
assert df.loc[is_toxic, "expression_type"].notna().all()
assert df.loc[~is_toxic, "expression_type"].isna().all()

print("Rows:", len(df))
display(df["toxicity"].value_counts().to_frame("count"))
display(df.loc[is_toxic, "expression_type"].value_counts().to_frame("count"))
print("DATASET VALIDATION: PASS")

In [ ]:
work = df.copy()
work["_stratum"] = np.where(
    work["toxicity"].eq("NON-TOXIC"),
    "NON_TOXIC",
    "TOXIC_" + work["expression_type"].astype(str)
)

train_df, temp_df = train_test_split(
    work,
    test_size=0.20,
    random_state=SEED,
    stratify=work["_stratum"]
)
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["_stratum"]
)

split_map = {}
for split_name, part in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    split_map.update({rid: split_name for rid in part["row_id"].tolist()})

full = df.copy()
full["split"] = full["row_id"].map(split_map)
assert full["split"].notna().all()

manifest = full[["row_id", "toxicity", "expression_type", "split"]].copy()
manifest.to_csv(SPLIT_DIR / "split_manifest.csv", index=False)

for split_name in ["train", "dev", "test"]:
    full[full["split"].eq(split_name)].to_csv(
        SPLIT_DIR / f"{split_name}.csv",
        index=False
    )

summary = []
for split_name in ["train", "dev", "test"]:
    p = full[full["split"].eq(split_name)]
    t = p[p["toxicity"].eq("TOXIC")]
    summary.append({
        "split": split_name,
        "total": len(p),
        "NON-TOXIC": int((p["toxicity"] == "NON-TOXIC").sum()),
        "TOXIC": int((p["toxicity"] == "TOXIC").sum()),
        "EXPLICIT": int((t["expression_type"] == "EXPLICIT").sum()),
        "IMPLICIT": int((t["expression_type"] == "IMPLICIT").sum()),
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

expected = {
    "train": (38936, 34248, 4688, 3999, 689),
    "dev":   (4867, 4281, 586, 500, 86),
    "test":  (4867, 4281, 586, 500, 86),
}

for _, r in summary_df.iterrows():
    got = tuple(int(r[c]) for c in ["total", "NON-TOXIC", "TOXIC", "EXPLICIT", "IMPLICIT"])
    assert got == expected[r["split"]], (r["split"], got)

metadata = {
    "dataset_sha256": DATASET_SHA256,
    "seed": SEED,
    "ratios": {"train": 0.80, "dev": 0.10, "test": 0.10},
    "stratification": ["NON_TOXIC", "TOXIC_EXPLICIT", "TOXIC_IMPLICIT"],
    "summary": summary,
}
(SPLIT_DIR / "split_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("MASTER SPLIT: PASS")

In [ ]:
train_ids = set(pd.read_csv(SPLIT_DIR / "train.csv")["row_id"])
dev_ids = set(pd.read_csv(SPLIT_DIR / "dev.csv")["row_id"])
test_ids = set(pd.read_csv(SPLIT_DIR / "test.csv")["row_id"])

assert train_ids.isdisjoint(dev_ids)
assert train_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(test_ids)
assert len(train_ids | dev_ids | test_ids) == 48670

print("NO OVERLAP: PASS")
print("UNION = 48,670: PASS")
print("ROOT:", ROOT)